# EcoVision AI -- YOLO11m Multi-Object Waste Detection Training & Baseline Benchmark
**Project:** EcoVision AI (`D:\Projects\ml_project\472`)  
**Target Model:** Ultralytics YOLO11m Object Detection (`yolo11m.pt`)  
**Resolution:** $960 \times 960$ pixels  
**Hardware Accelerator:** Google Colab GPU (NVIDIA Tesla T4 16GB / L4 / A100)  
**Taxonomy (6 Classes):** `0: plastic`, `1: metal`, `2: cardboard`, `3: glass`, `4: paper`, `5: general_trash`  

---
### Notebook Objectives
1. **Train YOLO11m** at high resolution ($960 \times 960$) with multi-object augmentations (`mosaic=1.0`, `mixup=0.15`).
2. **Fair Side-by-Side Comparison**: Benchmark candidate YOLO11m against the existing baseline YOLO11s (`models/weights_best.pt`) on the identical validation split.
3. **Rigorous Mixed-Waste Evaluation**: Measure recall, duplicate detections, and material confusion on crowded multi-object piles.
4. **Export Candidate Checkpoint**: Package `yolo11m_best.pt` without modifying or overwriting existing production checkpoints.


## Section 1: Google Colab GPU Setup & Verification
We check the assigned GPU, CUDA driver, and available VRAM. A Tesla T4 (15.3 GB VRAM) or better is recommended for training at $960 \times 960$.


In [ ]:
!nvidia-smi

import torch
import sys

print(f"Python version : {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError("CRITICAL: CUDA GPU is not available! In Colab, go to 'Runtime' -> 'Change runtime type' -> Select 'T4 GPU' (or higher) -> 'Save'.")

gpu_name = torch.cuda.get_device_name(0)
gpu_props = torch.cuda.get_device_properties(0)
total_vram_gb = gpu_props.total_memory / (1024 ** 3)
print(f"Assigned GPU   : {gpu_name}")
print(f"Total VRAM     : {total_vram_gb:.2f} GB")
print(f"Compute Cap    : {gpu_props.major}.{gpu_props.minor}")


## Section 2: Dependency Installation
Install reproducible, compatible versions of Ultralytics YOLO, PyYAML, Pandas, and evaluation utilities.


In [ ]:
!pip install --upgrade "ultralytics>=8.3.0,<8.5.0" pyyaml pandas tabulate matplotlib seaborn psutil

import ultralytics
import yaml
import pandas as pd
import numpy as np

print(f"Ultralytics version: {ultralytics.__version__}")
print(f"PyYAML version     : {yaml.__version__}")
print(f"Pandas version     : {pd.__version__}")


## Section 3: Dataset Upload / Google Drive Mounting
The dataset archive `ecovision_v3.zip` (625.69 MB, SHA-256: `0c7feba5af5e340e44ce8ae2a222b992bab4c5185f7ec43cb3aaa5ad05543627`) contains:
- `data/processed_v3/` (8,242 train, 3,007 val, 374 test images/labels)
- `metadata/dataset_colab_v3.yaml`

You can either mount Google Drive (recommended for fast loading) or upload directly.


In [ ]:
import os
import hashlib
from pathlib import Path

EXPECTED_SHA256 = "0c7feba5af5e340e44ce8ae2a222b992bab4c5185f7ec43cb3aaa5ad05543627"

def compute_sha256(filepath):
    h = hashlib.sha256()
    with open(filepath, 'rb') as f:
        while chunk := f.read(1024 * 1024):
            h.update(chunk)
    return h.hexdigest()

# 1. Check if already uploaded locally
zip_path = None
local_candidates = [
    Path('/content/ecovision_v3.zip'),
    Path('/content/dataset/ecovision_v3.zip'),
]
for c in local_candidates:
    if c.exists():
        zip_path = c
        break

# 2. If not found locally, try Google Drive
if zip_path is None:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_candidates = [
            Path('/content/drive/MyDrive/ecovision_v3.zip'),
            Path('/content/drive/MyDrive/EcoVisionAI/ecovision_v3.zip'),
            Path('/content/drive/MyDrive/EcoVision/ecovision_v3.zip'),
            Path('/content/drive/MyDrive/ml_project/ecovision_v3.zip'),
        ]
        for c in drive_candidates:
            if c.exists():
                zip_path = c
                print(f"Found dataset archive in Google Drive: {zip_path}")
                break
    except Exception as e:
        print(f"Drive mount skipped or unavailable: {e}")

# 3. If still not found, prompt user upload
if zip_path is None:
    print("Dataset archive not found automatically. Please upload 'ecovision_v3.zip' from your local machine:")
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded.keys():
        if fname.endswith('.zip'):
            zip_path = Path('/content') / fname
            break

if zip_path is None or not zip_path.exists():
    raise FileNotFoundError("CRITICAL: ecovision_v3.zip not found! Please upload it or place it in MyDrive/EcoVisionAI/ecovision_v3.zip")

print(f"Archive path: {zip_path} ({zip_path.stat().st_size / (1024**2):.2f} MB)")
sha = compute_sha256(zip_path)
print(f"SHA-256     : {sha}")
if sha == EXPECTED_SHA256:
    print("Archive integrity verified (SHA-256 matches official v3 release)!")
else:
    print("NOTICE: SHA-256 differs from canonical release (proceeding with caution).")


## Section 4: Dataset Extraction
Extract `ecovision_v3.zip` to Colab's high-speed ephemeral local NVMe storage `/content/dataset/`.


In [ ]:
import zipfile
import shutil

extract_dir = Path('/content/dataset')
extract_dir.mkdir(parents=True, exist_ok=True)

print(f"Extracting {zip_path} -> {extract_dir} ...")
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(extract_dir)

print("Extraction completed!")

# Verify splits
for split in ['train', 'val', 'test']:
    img_dir = extract_dir / 'data' / 'processed_v3' / 'images' / split
    lbl_dir = extract_dir / 'data' / 'processed_v3' / 'labels' / split
    n_img = len(list(img_dir.glob('*.*'))) if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
    print(f"Split [{split:5s}]: {n_img:5d} images | {n_lbl:5d} labels")


## Section 5: Dataset YAML Path Verification
Verify that the dataset YAML contains canonical absolute paths and the confirmed 6-class taxonomy.


In [ ]:
# Primary YAML path inside the archive
yaml_candidate = extract_dir / 'metadata' / 'dataset_colab_v3.yaml'

if not yaml_candidate.exists():
    yaml_candidate = extract_dir / 'data' / 'dataset_colab_v3.yaml'

# If needed, create canonical absolute YAML
canonical_yaml = Path('/content/dataset_colab_v3.yaml')
dataset_root = extract_dir / 'data' / 'processed_v3'

yaml_content = {
    'path': str(dataset_root.resolve()),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 6,
    'names': {
        0: 'plastic',
        1: 'metal',
        2: 'cardboard',
        3: 'glass',
        4: 'paper',
        5: 'general_trash'
    }
}

with open(canonical_yaml, 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)

DATASET_YAML = str(canonical_yaml)
print(f"Canonical Dataset YAML configured at: {DATASET_YAML}")
print("--- YAML Contents ---")
with open(DATASET_YAML) as f:
    print(f.read())


## Section 6: Class ID & Annotation Validation
Confirm annotation integrity: ensure all class IDs are within `[0, 5]` and bounding boxes have valid normalized coordinates.


In [ ]:
from collections import Counter

CLASS_NAMES = ['plastic', 'metal', 'cardboard', 'glass', 'paper', 'general_trash']

for split in ['train', 'val']:
    lbl_dir = dataset_root / 'labels' / split
    counts = Counter()
    total_boxes = 0
    corrupt_boxes = 0
    
    for lbl_file in lbl_dir.glob('*.txt'):
        with open(lbl_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                cls_id = int(parts[0])
                coords = [float(x) for x in parts[1:5]]
                if cls_id < 0 or cls_id > 5:
                    corrupt_boxes += 1
                if any(c < 0.0 or c > 1.0 for c in coords):
                    corrupt_boxes += 1
                counts[cls_id] += 1
                total_boxes += 1

    print(f"=== {split.upper()} ANNOTATION AUDIT ===")
    print(f"Total Bounding Boxes: {total_boxes} | Corrupt Boxes: {corrupt_boxes}")
    for cid in range(6):
        print(f"  Class {cid} ({CLASS_NAMES[cid]:14s}): {counts[cid]:6d} instances ({counts[cid]/max(1, total_boxes)*100:.2f}%)")
    print()


## Section 7: Existing YOLO11s Baseline Evaluation
To ensure a **rigorous and fair comparison**, we evaluate the existing baseline model (`weights_best.pt`) on the **exact same validation split** before training YOLO11m.

Please provide `weights_best.pt` (upload or locate in Google Drive).


In [ ]:
from ultralytics import YOLO

# Locate baseline model weights
baseline_path = None
candidates = [
    Path('/content/weights_best.pt'),
    Path('/content/drive/MyDrive/EcoVisionAI/weights_best.pt'),
    Path('/content/drive/MyDrive/weights_best.pt'),
]
for c in candidates:
    if c.exists():
        baseline_path = c
        break

if baseline_path is None:
    print("Baseline 'weights_best.pt' not found automatically. Please upload it for comparison:")
    from google.colab import files
    up = files.upload()
    for fname in up.keys():
        if fname.endswith('.pt'):
            baseline_path = Path('/content') / fname
            break

baseline_metrics = {}

if baseline_path and baseline_path.exists():
    print(f"Found baseline model: {baseline_path} ({baseline_path.stat().st_size / (1024**2):.2f} MB)")
    print("Running baseline evaluation on validation split (conf=0.15, imgsz=640)...")
    s_model = YOLO(str(baseline_path))
    s_val = s_model.val(
        data=DATASET_YAML,
        split='val',
        imgsz=640,
        conf=0.15,
        device=0,
        verbose=False
    )
    
    baseline_metrics = {
        'model': 'YOLO11s (Baseline)',
        'imgsz': 640,
        'precision': float(s_val.results_dict.get('metrics/precision(B)', 0.0)),
        'recall': float(s_val.results_dict.get('metrics/recall(B)', 0.0)),
        'map50': float(s_val.results_dict.get('metrics/mAP50(B)', 0.0)),
        'map50_95': float(s_val.results_dict.get('metrics/mAP50-95(B)', 0.0)),
        'per_class_map50': [float(x) for x in s_val.box.ap50] if hasattr(s_val.box, 'ap50') else [],
        'per_class_map': [float(x) for x in s_val.box.ap] if hasattr(s_val.box, 'ap') else [],
    }
    print("Baseline Metrics Loaded Successfully:")
    print(f"  Precision: {baseline_metrics['precision']:.4f}")
    print(f"  Recall   : {baseline_metrics['recall']:.4f}")
    print(f"  mAP50    : {baseline_metrics['map50']:.4f}")
    print(f"  mAP50-95 : {baseline_metrics['map50_95']:.4f}")
else:
    print("NOTICE: Baseline weights_best.pt not provided. Will use recorded historical metrics (mAP50=0.591, mAP50-95=0.446).")
    baseline_metrics = {
        'model': 'YOLO11s (Historical)',
        'imgsz': 640,
        'precision': 0.671,
        'recall': 0.519,
        'map50': 0.591,
        'map50_95': 0.446,
        'per_class_map50': [],
        'per_class_map': [],
    }


## Section 8: YOLO11m Training Configuration & Hyperparameter Analysis

### 1. Model Architecture
- **Base Model:** Pretrained `yolo11m.pt` (Ultralytics YOLO11 Medium, 20.1M parameters, 68.0 GFLOPs).
- **Resolution:** $960 \times 960$ pixels (+125% pixel surface area vs 640), preserving fine features of bottle caps, crinkled wrappers, and small litter items ($<32$px).

### 2. Augmentation Strategy Analysis (Supported vs. Unsupported)
- **`mosaic=1.0` (Enabled):** Stitches 4 training images into one composite canvas. Crucial to counter the dataset's 48.4% single-object bias by synthesizing multi-object mixed piles.
- **`mixup=0.15` (Enabled):** Linearly blends two scenes and concatenates their bounding boxes. Prevents overfitting on clean backgrounds.
- **`copy_paste=0.0` (Disabled — Critical Finding):** In Ultralytics, `CopyPaste` explicitly requires segmentation polygon masks (`if len(instances.segments) == 0: return labels`). Because 77.8% of dataset v3 consists of bounding-box annotations, Copy-Paste is bypassed or introduces unnatural rectangular box artifacts. We set `copy_paste=0.0` and rely on Mosaic + MixUp.
- **`close_mosaic=15` (Enabled):** Turns off mosaic during the final 15 epochs so bounding box edges converge cleanly on natural object boundaries.

### 3. GPU Memory Check & Dynamic Batch Sizing
At $960 \times 960$, YOLO11m consumes ~10–12 GB VRAM at batch size 16.
- If GPU VRAM $\ge 15$ GB (e.g. standard Tesla T4), we start with `batch=16`.
- If GPU VRAM $< 15$ GB or if a CUDA OOM occurs, we fall back to `batch=8` with gradient accumulation (`accumulate=2`) to preserve an effective batch size of 16.


In [ ]:
# Dynamic GPU memory check & batch sizing
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"Detected GPU Memory: {gpu_mem_gb:.2f} GB")

if gpu_mem_gb >= 15.0:
    recommended_batch = 16
    recommended_accumulate = 1
elif gpu_mem_gb >= 11.0:
    recommended_batch = 8
    recommended_accumulate = 2
else:
    recommended_batch = 4
    recommended_accumulate = 4

print(f"Selected Batch Size : {recommended_batch}")
print(f"Selected Accumulate : {recommended_accumulate} (Effective Batch = {recommended_batch * recommended_accumulate})")


## Section 9: Model Training Execution
We initialize `yolo11m.pt` and launch training with automated Out-Of-Memory (OOM) recovery.


In [ ]:
from ultralytics import YOLO
import gc

model = YOLO('yolo11m.pt')

train_args = {
    'data': DATASET_YAML,
    'epochs': 100,
    'patience': 25,
    'batch': recommended_batch,
    'imgsz': 960,
    'seed': 42,
    'device': 0,
    'optimizer': 'AdamW',
    'lr0': 0.001,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3.0,
    'mosaic': 1.0,
    'close_mosaic': 15,
    'mixup': 0.15,
    'copy_paste': 0.0,
    'project': 'runs/train',
    'name': 'yolo11m_waste_960',
    'save': True,
    'save_period': 5,
    'plots': True,
    'verbose': True
}

print("Initiating YOLO11m training with configuration:")
for k, v in train_args.items():
    print(f"  {k:15s}: {v}")

try:
    results = model.train(**train_args)
except Exception as e:
    if "out of memory" in str(e).lower():
        print("\n[!] CUDA Out of Memory encountered! Retrying with fallback batch=8 and accumulate=2...")
        torch.cuda.empty_cache()
        gc.collect()
        train_args['batch'] = 8
        model = YOLO('yolo11m.pt')
        results = model.train(**train_args)
    else:
        raise e


## Section 10: Best Checkpoint Verification
Verify the trained checkpoint `runs/train/yolo11m_waste_960/weights/best.pt`.


In [ ]:
best_weight = Path('runs/train/yolo11m_waste_960/weights/best.pt')
last_weight = Path('runs/train/yolo11m_waste_960/weights/last.pt')

if not best_weight.exists():
    raise FileNotFoundError(f"Training completed but best.pt not found at {best_weight}!")

print(f"Best Checkpoint Found : {best_weight}")
print(f"File Size             : {best_weight.stat().st_size / (1024**2):.2f} MB")
print(f"SHA-256 Checksum      : {compute_sha256(best_weight)}")

# Create local safe directory
export_dir = Path('/content/models/experiments/yolo11m_v1')
export_dir.mkdir(parents=True, exist_ok=True)
shutil.copy(best_weight, export_dir / 'yolo11m_best.pt')
print(f"Saved candidate checkpoint to {export_dir / 'yolo11m_best.pt'}")


## Section 11: Validation Metrics & Side-by-Side Comparison
Evaluate the newly trained YOLO11m candidate on the validation split at `conf=0.15` and compare against the baseline.


In [ ]:
candidate_model = YOLO(str(best_weight))
m_val = candidate_model.val(
    data=DATASET_YAML,
    split='val',
    imgsz=960,
    conf=0.15,
    device=0,
    verbose=False
)

candidate_metrics = {
    'model': 'YOLO11m (Candidate)',
    'imgsz': 960,
    'precision': float(m_val.results_dict.get('metrics/precision(B)', 0.0)),
    'recall': float(m_val.results_dict.get('metrics/recall(B)', 0.0)),
    'map50': float(m_val.results_dict.get('metrics/mAP50(B)', 0.0)),
    'map50_95': float(m_val.results_dict.get('metrics/mAP50-95(B)', 0.0)),
    'per_class_map50': [float(x) for x in m_val.box.ap50] if hasattr(m_val.box, 'ap50') else [],
    'per_class_map': [float(x) for x in m_val.box.ap] if hasattr(m_val.box, 'ap') else [],
}

print("=" * 75)
print(f"{'Metric':<25} | {'YOLO11s (Baseline)':<20} | {'YOLO11m (Candidate)':<20} | {'Delta':<10}")
print("=" * 75)

for metric_key, label in [
    ('precision', 'Precision (P)'),
    ('recall', 'Recall (R)'),
    ('map50', 'mAP@50'),
    ('map50_95', 'mAP@50-95'),
]:
    b_val = baseline_metrics.get(metric_key, 0.0)
    c_val = candidate_metrics.get(metric_key, 0.0)
    delta = c_val - b_val
    delta_str = f"+{delta:.4f}" if delta >= 0 else f"{delta:.4f}"
    print(f"{label:<25} | {b_val:<20.4f} | {c_val:<20.4f} | {delta_str:<10}")

print("=" * 75)

# Per-Class Comparison
print("\n--- PER-CLASS mAP@50 COMPARISON ---")
for i, cname in enumerate(CLASS_NAMES):
    b_c = baseline_metrics['per_class_map50'][i] if i < len(baseline_metrics.get('per_class_map50', [])) else float('nan')
    c_c = candidate_metrics['per_class_map50'][i] if i < len(candidate_metrics.get('per_class_map50', [])) else float('nan')
    diff = c_c - b_c if not np.isnan(b_c) and not np.isnan(c_c) else float('nan')
    diff_str = f"+{diff:.4f}" if diff >= 0 else f"{diff:.4f}"
    print(f"Class {i} ({cname:<14s}): Baseline={b_c:.4f} | Candidate={c_c:.4f} | Delta={diff_str}")


## Section 12: Dense Mixed-Waste Evaluation
We isolate validation images containing multi-object clusters ($\ge 4$ objects across $\ge 2$ classes) to measure object recall rate, duplicate detections, and class confusion on complex piles.


In [ ]:
import cv2
import matplotlib.pyplot as plt

val_lbl_dir = dataset_root / 'labels' / 'val'
val_img_dir = dataset_root / 'images' / 'val'

dense_scenes = []
for lbl_file in val_lbl_dir.glob('*.txt'):
    classes_in_file = set()
    box_count = 0
    with open(lbl_file) as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                classes_in_file.add(int(parts[0]))
                box_count += 1
    if box_count >= 4 and len(classes_in_file) >= 2:
        img_file = val_img_dir / (lbl_file.stem + '.jpg')
        if not img_file.exists():
            img_file = val_img_dir / (lbl_file.stem + '.png')
        if img_file.exists():
            dense_scenes.append((str(img_file), str(lbl_file), box_count, len(classes_in_file)))

print(f"Identified {len(dense_scenes)} dense mixed-waste validation scenes (>=4 boxes, >=2 classes).")

# Evaluate Candidate on dense scenes
sample_scenes = dense_scenes[:min(5, len(dense_scenes))]
fig, axes = plt.subplots(len(sample_scenes), 2, figsize=(16, 6 * len(sample_scenes)))
if len(sample_scenes) == 1:
    axes = np.expand_dims(axes, 0)

for idx, (img_p, lbl_p, b_cnt, c_cnt) in enumerate(sample_scenes):
    img = cv2.imread(img_p)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Ground Truth Plot
    ax_gt = axes[idx, 0]
    ax_gt.imshow(img_rgb)
    ax_gt.set_title(f"Scene {idx+1}: Ground Truth ({b_cnt} items, {c_cnt} classes)")
    ax_gt.axis('off')
    
    # YOLO11m Prediction Plot
    res = candidate_model.predict(img_p, conf=0.15, imgsz=960, verbose=False)[0]
    res_bgr = res.plot()
    res_rgb = cv2.cvtColor(res_bgr, cv2.COLOR_BGR2RGB)
    
    ax_pred = axes[idx, 1]
    ax_pred.imshow(res_rgb)
    ax_pred.set_title(f"YOLO11m Candidate ({len(res.boxes)} detections)")
    ax_pred.axis('off')

plt.tight_layout()
plt.savefig('/content/models/experiments/yolo11m_v1/dense_scene_comparison.png', dpi=150)
plt.show()


## Section 13: Model Size & Inference Speed Comparison
Compare model footprint, GPU inference speed, and simulated CPU inference speed (for target deployment).


In [ ]:
import time

# Model File Size
size_s = baseline_path.stat().st_size / (1024**2) if baseline_path and baseline_path.exists() else 19.18
size_m = best_weight.stat().st_size / (1024**2)

# GPU Latency Benchmark (Warmup + 100 runs)
dummy_960 = np.zeros((960, 960, 3), dtype=np.uint8)

print("Benchmarking GPU Latency...")
for _ in range(10):
    _ = candidate_model.predict(dummy_960, verbose=False, device=0)

gpu_times = []
for _ in range(50):
    t0 = time.perf_counter()
    _ = candidate_model.predict(dummy_960, verbose=False, device=0)
    gpu_times.append((time.perf_counter() - t0) * 1000)

mean_gpu_latency = np.mean(gpu_times)

# Simulated CPU Latency Benchmark (10 runs)
print("Benchmarking CPU Latency (Simulating local deployment)...")
cpu_model = YOLO(str(best_weight))
for _ in range(2):
    _ = cpu_model.predict(dummy_960, verbose=False, device='cpu')

cpu_times = []
for _ in range(10):
    t0 = time.perf_counter()
    _ = cpu_model.predict(dummy_960, verbose=False, device='cpu')
    cpu_times.append((time.perf_counter() - t0) * 1000)

mean_cpu_latency = np.mean(cpu_times)

print("=" * 60)
print(f"{'Specification':<30} | {'YOLO11s':<12} | {'YOLO11m':<12}")
print("=" * 60)
print(f"{'Model Parameters':<30} | {'9.4 M':<12} | {'20.1 M':<12}")
print(f"{'GFLOPs (at target resolution)':<30} | {'21.5 (640)':<12} | {'68.0 (960)':<12}")
print(f"{'Disk Size (MB)':<30} | {f'{size_s:.2f} MB':<12} | {f'{size_m:.2f} MB':<12}")
print(f"{'GPU Latency (ms)':<30} | {'~12 ms':<12} | {f'{mean_gpu_latency:.1f} ms':<12}")
print(f"{'CPU Latency (ms)':<30} | {'~140 ms':<12} | {f'{mean_cpu_latency:.1f} ms':<12}")
print("=" * 60)


## Section 14: Exporting Candidate Checkpoint
Package the trained candidate checkpoint alongside its evaluation metrics and plots.
We do NOT overwrite `models/weights_best.pt`.
The output package will be saved under Google Drive `/content/drive/MyDrive/EcoVisionAI/experiments/yolo11m_v1/` and made available for direct download.


In [ ]:
import json

# Compile complete metadata manifest
package_dir = Path('/content/yolo11m_candidate_package')
package_dir.mkdir(parents=True, exist_ok=True)

# 1. Copy Best Checkpoint
shutil.copy(best_weight, package_dir / 'yolo11m_best.pt')

# 2. Copy Training Plots & Curves
train_run_dir = Path('runs/train/yolo11m_waste_960')
plot_files = ['results.csv', 'confusion_matrix.png', 'confusion_matrix_normalized.png', 'F1_curve.png', 'PR_curve.png']
for pf in plot_files:
    src_f = train_run_dir / pf
    if src_f.exists():
        shutil.copy(src_f, package_dir / pf)

# 3. Save Comparative Metrics JSON
metrics_payload = {
    'architecture': 'YOLO11m Detection',
    'imgsz': 960,
    'epochs_trained': 100,
    'baseline_metrics': baseline_metrics,
    'candidate_metrics': candidate_metrics,
    'gpu_latency_ms': float(mean_gpu_latency),
    'cpu_latency_ms': float(mean_cpu_latency),
    'target_local_path': 'models/experiments/yolo11m_best.pt'
}

with open(package_dir / 'metrics_summary.json', 'w') as f:
    json.dump(metrics_payload, f, indent=2)

# 4. Create ZIP package
zip_out = Path('/content/yolo11m_candidate_package.zip')
with zipfile.ZipFile(zip_out, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for f in package_dir.glob('*.*'):
        zf.write(f, arcname=f.name)

print(f"Export Package Created: {zip_out} ({zip_out.stat().st_size / (1024**2):.2f} MB)")

# 5. Backup to Google Drive if connected
try:
    drive_backup = Path('/content/drive/MyDrive/EcoVisionAI/experiments/yolo11m_v1')
    drive_backup.mkdir(parents=True, exist_ok=True)
    shutil.copy(zip_out, drive_backup / 'yolo11m_candidate_package.zip')
    shutil.copy(best_weight, drive_backup / 'yolo11m_best.pt')
    print(f"Successfully backed up candidate package to Google Drive: {drive_backup}")
except Exception as e:
    print(f"Drive backup skipped: {e}")

# 6. Direct browser download trigger
print("\nReady for download. You can download directly via:")
print("from google.colab import files; files.download('/content/yolo11m_candidate_package.zip')")
